# 해안쓰레기 YOLO-seg 학습 (Google Colab)

**사용법**: 위 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 로 바꾼 뒤, 셀을 위에서부터 차례로 실행(`Shift+Enter`).
`# ✏️` 표시가 있는 줄만 상황에 맞게 고치면 됩니다.

흐름: ① GPU 확인 → ② 코드 받기 → ③ Drive 연결 + 데이터 위치 지정 → ④ 라벨 형식 자동 점검 → ⑤ YOLO-seg 형식으로 변환·분할
→ ⑥ 학습 → ⑦ 평가(재현율 중심) → ⑧ 예측 예시 → ⑨ 가중치 Drive 저장

> 데이터는 **Google Drive** 에 올려 두세요 (예: `내 드라이브/hackathon/data/` 아래에 이미지와 라벨을 그대로 압축 해제).
> Colab 무료 GPU 는 한 세션 최대 수 시간이라, 학습 중간 결과(`last.pt`)를 Drive 에 저장하고 이어서 학습할 수 있게 해 두었습니다.

## ① GPU 확인

In [ ]:
!nvidia-smi -L || echo "GPU 없음 → 런타임 → 런타임 유형 변경 → T4 GPU"
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## ② 코드(litter3d) 받기와 설치

In [ ]:
import os, sys
REPO = "https://github.com/jihu0629/Aerodrone-Hakaton.git"   # ✏️ 저장소 주소
if not os.path.exists("/content/Aerodrone-Hakaton"):
    !git clone -q {REPO} /content/Aerodrone-Hakaton
else:
    !cd /content/Aerodrone-Hakaton && git pull -q
%cd /content/Aerodrone-Hakaton
!pip install -q ultralytics opencv-python-headless shapely
sys.path.insert(0, "/content/Aerodrone-Hakaton")
import ultralytics; ultralytics.checks()
from litter3d.classes import class_names, KO_NAMES
print("클래스 13종:", [f"{i}:{n}({KO_NAMES[n]})" for i, n in enumerate(class_names())])

## ③ Google Drive 연결 + 데이터 위치

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/hackathon/data"      # ✏️ 기업 데이터를 풀어 둔 폴더 (이미지 + 라벨)
WORK_DIR = "/content/drive/MyDrive/hackathon/work"      # ✏️ 학습 결과(가중치)를 저장할 폴더
os.makedirs(WORK_DIR, exist_ok=True)
assert os.path.isdir(DATA_DIR), f"폴더가 없습니다: {DATA_DIR}"
!ls -la "{DATA_DIR}" | head -20

### (선택) zip 으로 올렸다면 여기서 풀기
Drive 에서 바로 읽으면 느리므로 Colab 로컬(/content)로 복사·해제하는 편이 학습이 훨씬 빠릅니다.

In [ ]:
ZIP_PATH = ""   # ✏️ 예: "/content/drive/MyDrive/hackathon/data.zip"  (없으면 빈칸)
if ZIP_PATH:
    !mkdir -p /content/data && unzip -q -o "{ZIP_PATH}" -d /content/data
    DATA_DIR = "/content/data"
    !ls "{DATA_DIR}" | head

## ④ 라벨 형식 자동 점검
결과의 `format` 이 `coco / yolo / labelme / per_image_json / voc` 중 무엇인지, `samples` 의 라벨 파일 앞부분을 확인하세요.
`unknown` 이 나오면 `samples` 내용을 복사해서 알려주시면 변환 코드를 맞춰 드립니다.

In [ ]:
import json
from litter3d.dataset import inspect_dataset
info = inspect_dataset(DATA_DIR)
print("형식:", info["format"], "| 이미지:", info["n_images"], "장 | 이미지 크기:", info.get("image_shape"))
print("확장자:", info["extensions"])
for s in info["samples"]:
    print("\n=====", s["file"], "=====\n", s["head"])

## ⑤ YOLO-seg 형식으로 변환 + train/val 분할
- 클래스명은 `litter3d.classes.map_class()` 가 우리 13종으로 자동 매핑합니다. 매핑이 틀리면 아래 `CLASS_MAP` 에 직접 적으세요.
- `per_image_json` 인데 키 이름을 못 찾으면 `HINTS` 에 키를 지정하세요 (④ 출력의 라벨 파일 앞부분 참고).
- 박스만 있는 라벨은 사각 폴리곤으로 변환됩니다(`skipped.bbox_only` 개수). 정밀 마스크가 필요하면 나중에 SAM 으로 보정합니다.

In [ ]:
from litter3d.dataset import prepare_dataset, KeyHints

CLASS_MAP = {                     # ✏️ 원본 클래스명 → 우리 클래스 (자동 매핑이 틀릴 때만)
    # "스티로폼": "styrofoam_fragment",
    # "페트병": "pet_bottle",
}
HINTS = KeyHints(                 # ✏️ per_image_json 에서 키를 직접 지정할 때만 (None = 자동)
    objects=None, label=None, polygon=None, bbox=None, image_file=None, class_map=CLASS_MAP,
)
YOLO_DIR = "/content/yolo_ds"
FORMAT = None                     # ✏️ 자동 판별이 틀리면 "coco" / "yolo" / "labelme" / "per_image_json" / "voc"

report = prepare_dataset(DATA_DIR, YOLO_DIR, fmt=FORMAT, hints=HINTS, val_ratio=0.2, copy=True)
print(json.dumps(report, ensure_ascii=False, indent=2))
DATA_YAML = report["data_yaml"]

### 변환 결과 눈으로 확인
`orig_label_counts`(원본 클래스명별 개수)와 `per_class`(우리 클래스별 개수)를 비교해서 매핑이 맞는지 보세요.
아래 셀은 라벨을 이미지 위에 그려서 3장 보여줍니다. 폴리곤이 물체와 어긋나면 좌표 순서(x,y)나 정규화 문제입니다.

In [ ]:
import cv2, random, numpy as np
from pathlib import Path
from matplotlib import pyplot as plt
names = class_names()
imgs = sorted(Path(YOLO_DIR, "images/train").glob("*"))
random.seed(0)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, im in zip(axes, random.sample(imgs, min(3, len(imgs)))):
    img = cv2.imread(str(im))[..., ::-1].copy(); h, w = img.shape[:2]
    for line in Path(YOLO_DIR, "labels/train", im.stem + ".txt").read_text().splitlines():
        p = line.split(); cls = int(p[0]); pts = np.array(p[1:], float).reshape(-1, 2) * [w, h]
        cv2.polylines(img, [pts.astype(np.int32)], True, (255, 0, 0), 2)
        cv2.putText(img, names[cls], tuple(pts[0].astype(int)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    ax.imshow(img); ax.set_title(im.name); ax.axis("off")
plt.show()

## ⑥ 학습
- `MODEL`: `yolo11n-seg.pt`(빠름) → `yolo11s-seg.pt`(정확) → `yolo11m-seg.pt`(T4 에서 imgsz 1024 는 메모리 주의).
- `IMGSZ`: 드론 사진은 작은 물체가 많아 1024 이상 권장. 원본이 8192×6144 처럼 크면 ⑤ 전에 타일로 자르는 게 좋습니다(아래 참고).
- 세션이 끊기면 `RESUME = True` 로 바꾸고 다시 실행하면 `last.pt` 에서 이어서 학습합니다.

In [ ]:
from ultralytics import YOLO

MODEL  = "yolo11s-seg.pt"   # ✏️
EPOCHS = 80                 # ✏️ 데이터가 적으면 100+, 많으면 50 정도
IMGSZ  = 1024               # ✏️
BATCH  = 8                  # ✏️ 메모리 부족(CUDA out of memory)이면 4 로
RUN_NAME = "litter_seg"
RESUME = False              # ✏️ 끊긴 학습 이어서 하려면 True

last = f"{WORK_DIR}/runs/{RUN_NAME}/weights/last.pt"
if RESUME and os.path.exists(last):
    model = YOLO(last)
    results = model.train(resume=True)
else:
    model = YOLO(MODEL)
    results = model.train(
        data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        project=f"{WORK_DIR}/runs", name=RUN_NAME, exist_ok=True,
        patience=20,            # 20 에폭 동안 개선 없으면 조기 종료
        close_mosaic=10,
        degrees=180, flipud=0.5, fliplr=0.5,   # 드론 수직 촬영은 회전·상하반전이 자연스러움
        scale=0.3, hsv_h=0.02, hsv_s=0.5, hsv_v=0.4,
        cos_lr=True, seed=0, plots=True,
    )
BEST = f"{WORK_DIR}/runs/{RUN_NAME}/weights/best.pt"
print("best.pt →", BEST)

## ⑦ 평가
무게 추정에서는 **재현율(Recall)** 이 특히 중요합니다. 놓친 쓰레기는 무게에서 통째로 빠지기 때문입니다 (Andriolo et al. 2024 에서 AI 검출 누락이 -50 % 오차의 주원인).
클래스별 R 이 낮은 클래스는 데이터 보강 대상입니다.

In [ ]:
model = YOLO(BEST)
metrics = model.val(data=DATA_YAML, imgsz=IMGSZ, split="val", plots=True, conf=0.25)
print(f"전체: 마스크 mAP50 {metrics.seg.map50:.3f} | mAP50-95 {metrics.seg.map:.3f} | P {metrics.seg.mp:.3f} | R {metrics.seg.mr:.3f}")
print("\n클래스별 (마스크 기준)")
print(f"{'클래스':22} {'정밀도':>7} {'재현율':>7} {'mAP50':>7}")
for i, c in enumerate(metrics.ap_class_index):
    n = names[int(c)]
    print(f"{n+'('+KO_NAMES[n]+')':22} {metrics.seg.p[i]:7.3f} {metrics.seg.r[i]:7.3f} {metrics.seg.ap50[i]:7.3f}")

In [ ]:
# 학습 곡선·혼동행렬 이미지
from IPython.display import Image, display
run_dir = f"{WORK_DIR}/runs/{RUN_NAME}"
for f in ["results.png", "confusion_matrix_normalized.png"]:
    p = Path(run_dir, f)
    if p.exists(): display(Image(str(p), width=800))

## ⑧ 예측 예시 (val 이미지 3장)

In [ ]:
val_imgs = sorted(Path(YOLO_DIR, "images/val").glob("*"))[:3]
res = model.predict([str(p) for p in val_imgs], imgsz=IMGSZ, conf=0.25, verbose=False)
fig, axes = plt.subplots(1, len(res), figsize=(6 * len(res), 6))
for ax, r in zip(np.atleast_1d(axes), res):
    ax.imshow(r.plot()[..., ::-1]); ax.axis("off"); ax.set_title(f"{len(r.boxes)}개 검출")
plt.show()

## ⑨ 가중치 저장·다운로드
`best.pt` 는 이미 Drive(`WORK_DIR/runs/.../weights/`)에 있습니다. 로컬 PC 파이프라인에서는 이렇게 씁니다.
```
python scripts/13_estimate_mass.py --dsm dsm.tif --ortho ortho.tif --weights best.pt
```

In [ ]:
import shutil
shutil.copy2(BEST, f"{WORK_DIR}/best_{RUN_NAME}.pt")
print("저장:", f"{WORK_DIR}/best_{RUN_NAME}.pt")
from google.colab import files
# files.download(BEST)   # ✏️ 브라우저로 바로 내려받으려면 주석 해제

---
## 부록 A. 큰 사진을 타일로 자르기 (원본이 4000 px 이상일 때)
8192×6144 사진을 1024 로 줄이면 5 cm 쓰레기가 6 px 이 되어 못 잡습니다. 학습 전에 1024×1024 타일(겹침 128)로 자르고 라벨도 같이 자릅니다.
⑤ 실행 후 이 셀을 돌리면 `YOLO_DIR` 이 타일 데이터셋으로 바뀝니다. 그다음 ⑥ 부터 다시 실행하세요.

In [ ]:
TILE, OVERLAP = 1024, 128        # ✏️
from shapely.geometry import Polygon, box as sbox
def tile_split(src, dst):
    for split in ("train", "val"):
        (Path(dst, "images", split)).mkdir(parents=True, exist_ok=True)
        (Path(dst, "labels", split)).mkdir(parents=True, exist_ok=True)
        for im in sorted(Path(src, "images", split).glob("*")):
            img = cv2.imread(str(im)); h, w = img.shape[:2]
            polys = []
            lab = Path(src, "labels", split, im.stem + ".txt")
            if lab.exists():
                for line in lab.read_text().splitlines():
                    p = line.split()
                    if len(p) >= 7:
                        polys.append((int(p[0]), Polygon(np.array(p[1:], float).reshape(-1, 2) * [w, h])))
            step = TILE - OVERLAP
            for y in range(0, max(h - OVERLAP, 1), step):
                for x in range(0, max(w - OVERLAP, 1), step):
                    x1, y1 = min(x + TILE, w), min(y + TILE, h)
                    tw, th = x1 - x, y1 - y
                    if tw < TILE // 2 or th < TILE // 2: continue
                    win = sbox(x, y, x1, y1); lines = []
                    for cls, pg in polys:
                        if not pg.is_valid: pg = pg.buffer(0)
                        inter = pg.intersection(win)
                        if inter.is_empty or inter.area < 0.3 * pg.area and inter.area < 400: continue
                        geoms = [inter] if inter.geom_type == "Polygon" else [g for g in getattr(inter, "geoms", []) if g.geom_type == "Polygon"]
                        for g in geoms:
                            pts = np.array(g.exterior.coords)[:-1] - [x, y]
                            lines.append(f"{cls} " + " ".join(f"{v:.6f}" for v in (pts / [tw, th]).ravel()))
                    if not lines and split == "train" and random.random() < 0.7:   # 빈 타일은 30 % 만 유지
                        continue
                    name = f"{im.stem}_{y}_{x}"
                    cv2.imwrite(str(Path(dst, "images", split, name + ".jpg")), img[y:y1, x:x1], [cv2.IMWRITE_JPEG_QUALITY, 95])
                    Path(dst, "labels", split, name + ".txt").write_text("\n".join(lines) + ("\n" if lines else ""))
    Path(dst, "data.yaml").write_text(Path(src, "data.yaml").read_text().replace(str(Path(src).resolve()), str(Path(dst).resolve())))
TILED = "/content/yolo_tiles"
tile_split(YOLO_DIR, TILED)
YOLO_DIR, DATA_YAML = TILED, str(Path(TILED, "data.yaml"))
from litter3d.dataset import yolo_class_histogram
print("타일 train:", len(list(Path(TILED, "images/train").glob("*"))), "val:", len(list(Path(TILED, "images/val").glob("*"))))
print(yolo_class_histogram(Path(TILED, "labels/train")))

## 부록 B. 색 기반 베이스라인과 비교 (Kako 2020 방식)
발표에서 "색만 보는 기존 방식 vs 우리 CNN 분할" 을 비교할 때 씁니다. val 이미지에서 두 방법의 픽셀 IoU 를 계산합니다.

In [ ]:
from litter3d.segment import color_baseline
def gt_mask(stem, shape):
    m = np.zeros(shape[:2], bool)
    lab = Path(YOLO_DIR, "labels/val", stem + ".txt")
    if lab.exists():
        for line in lab.read_text().splitlines():
            p = line.split(); pts = (np.array(p[1:], float).reshape(-1, 2) * [shape[1], shape[0]]).astype(np.int32)
            cv2.fillPoly(m.view(np.uint8), [pts], 1)
    return m
def iou(a, b): return (a & b).sum() / max((a | b).sum(), 1)
ious_c, ious_y = [], []
for im in sorted(Path(YOLO_DIR, "images/val").glob("*"))[:30]:
    img = cv2.imread(str(im)); gt = gt_mask(im.stem, img.shape)
    if gt.sum() == 0: continue
    mc = np.zeros_like(gt)
    for _, m, _ in color_baseline(img): mc |= m
    my = np.zeros_like(gt)
    r = model.predict(img, imgsz=IMGSZ, conf=0.25, verbose=False)[0]
    if r.masks is not None:
        for m in r.masks.data.cpu().numpy():
            my |= cv2.resize(m.astype(np.uint8), (img.shape[1], img.shape[0])) > 0
    ious_c.append(iou(mc, gt)); ious_y.append(iou(my, gt))
print(f"픽셀 IoU 평균 — 색 기반: {np.mean(ious_c):.3f} | YOLO-seg: {np.mean(ious_y):.3f}  (val {len(ious_y)}장)")